In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import pickle
from datetime import datetime

In [2]:
data = pd.read_excel("Data/dataset_comp_1222_without_ceg_ogn.xlsx")

## 2023–mid 2025 backtesting

In [3]:
data = pd.read_excel("Data/dataset_comp_1222_without_ceg_ogn.xlsx")

In [4]:
# # Define the tickers and date range
# tickers = list(data['SYMBOL'].values) # Add more tickers as needed
# start_date = "2022-12-30"
# end_date = "2025-07-01"

# # Download the data
# yahoo_data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_bt = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_bt[tickers[0]] = yahoo_data['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_bt[ticker] = yahoo_data[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# print(adj_close_bt.head())

In [5]:
# #print(data.loc[data['SYMBOL'] == 'BF.B'])
# # Define the tickers and date range
# tickers = ['BF-B'] # Add more tickers as needed
# start_date = "2022-12-30"
# end_date = "2025-07-01"

# # Download the data
# data_bfb = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_bfb = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_bfb[tickers[0]] = data_bfb['BF-B']['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_bfb[ticker] = data_bfb[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# #print(adj_close_bfb.head())
# adj_close_bt['BF.B'] = adj_close_bfb.values

Redownload failed downloads:

In [6]:
# # Define the tickers and date range
# tickers = ['HUM'] # Add more tickers as needed
# start_date = "2022-12-30"
# end_date = "2025-07-01"

# # Download the data
# data_redownload = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_redownload = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_redownload[tickers[0]] = data_redownload[tickers[0]]['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_redownload[ticker] = data_redownload[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# adj_close_bt[tickers] = adj_close_redownload.values

In [7]:
# adj_close_bt.to_excel("Data/adj_price_yahoo_comp_1222_2325.xlsx")

In [8]:
adj_close_bt = pd.read_excel("Data/adj_price_yahoo_comp_1222_2325.xlsx")
adj_close_bt.index = adj_close_bt.Date
adj_close_bt.drop(columns='Date', inplace=True)

In [9]:
nan_columns = adj_close_bt.columns[adj_close_bt.isna().any()]
print(nan_columns)

Index(['DISH', 'ATVI', 'PXD', 'SIVBQ', 'MRO', 'SBNY', 'DFS', 'CTLT', 'WRK'], dtype='object')


In [10]:
price_historical = pd.read_excel('Data/price_div_comp_1222_2325.xlsm', sheet_name='CLOSE PRICE', header = 4)

price_historical.columns = price_historical.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()

In [11]:
price_historical = pd.read_excel('Data/price_div_comp_1222_2325.xlsm', sheet_name='CLOSE PRICE', header = 4)

price_historical.columns = price_historical.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()

price_historical = price_historical.iloc[:-1]

div_rate = pd.read_excel('Data/price_div_comp_1222_2325.xlsm', sheet_name='DIV RATE', header = 4)
div_rate.columns = div_rate.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_rate = div_rate.iloc[:-1]

div_date_historical = pd.read_excel('Data/price_div_comp_1222_2325.xlsm', sheet_name='DIV DATE', header = 4)
div_date_historical.columns = div_date_historical.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_date_historical = div_date_historical.iloc[:-1]

div_date_historical.columns = price_historical.columns
div_rate.columns = price_historical.columns

price_historical.index = price_historical.Code
div_rate.index = div_rate.Code
div_date_historical.index = div_date_historical.Code

price_historical = price_historical.iloc[:, 1:]
div_rate = div_rate.iloc[:, 1:]
div_date_historical = div_date_historical.iloc[:, 1:]

# --- Step 1: Calculate adjustment factors
adj_factors = pd.DataFrame(1.0, index=price_historical.index, columns=price_historical.columns)

for company in price_historical.columns:
    for i in range(1, len(price_historical)):
        date = price_historical.index[i]
        prev_date = price_historical.index[i - 1]

        # If ex-dividend happens on this day
        if pd.notna(div_date_historical.at[date, company]):
            div = div_rate.at[date, company]
            price_prev = price_historical.at[prev_date, company]
            if price_prev and price_prev != 0:
                factor = (price_prev - div) / price_prev
                adj_factors.at[date, company] = factor

# --- Step 2: Calculate cumulative adjustment factors in reverse (like Yahoo)
cum_factors = adj_factors.iloc[::-1].cumprod().iloc[::-1]

# --- Step 3: Build adjusted prices
adjusted_prices_calculated = price_historical * cum_factors

#print(data.loc[data['SYMBOL'] == 'WRK', 'NAME'])
#print(div_rate['96699P'].unique())
#print(price_historical['96699P'])
#print(adjusted_prices_calculated['96699P'])
adj_close_bt['WRK'] = adjusted_prices_calculated.loc[adj_close_bt.index, '96699P'].values

#print(data.loc[data['SYMBOL'] == 'CTLT', 'NAME'])
#print(div_rate['8866F3'].unique())
#print(price_historical['8866F3'])
#print(adjusted_prices_calculated['8866F3'])
adj_close_bt['CTLT'] = adjusted_prices_calculated.loc[adj_close_bt.index, ['8866F3']].values

#print(data.loc[data['SYMBOL'] == 'MRO', 'NAME'])
#print(div_rate['544682'].unique())
#print(price_historical['544682'])
#print(adjusted_prices_calculated['544682'])
adj_close_bt['MRO'] = adjusted_prices_calculated.loc[adj_close_bt.index, '544682'].values

#print(data.loc[data['SYMBOL'] == 'PXD', 'NAME'])
#print(div_rate['895705'].unique())
#print(price_historical['895705'])
adj_close_bt['PXD'] = adjusted_prices_calculated.loc[adj_close_bt.index, '895705'].values

#print(data.loc[data['SYMBOL'] == 'SBNY', ['NAME','TYPE']])
#print(div_rate['28709C'].unique())
#print(price_historical['28709C'])
adj_close_bt['SBNY'] = adjusted_prices_calculated.loc[adj_close_bt.index, '28709C'].values

#print(data.loc[data['SYMBOL'] == 'ATVI', ['NAME','TYPE']])
#print(div_rate['312367'].unique())
#print(price_historical['312367'])
adj_close_bt['ATVI'] = adjusted_prices_calculated.loc[adj_close_bt.index, '312367'].values

#print(data.loc[data['SYMBOL'] == 'DISH', ['NAME','TYPE']])
#print(div_rate['135448'].unique())
#print(price_historical['135448'])
adj_close_bt['DISH'] = adjusted_prices_calculated.loc[adj_close_bt.index, '135448'].values

#print(data.loc[data['SYMBOL'] == 'SIVBQ', ['NAME','TYPE']])
#print(div_rate['518628'].unique())
#print(price_historical['518628'])
adj_close_bt['SIVBQ'] = adjusted_prices_calculated.loc[adj_close_bt.index, '518628'].values

print(div_rate['50642F'].unique())
print(price_historical['50642F'])
adj_close_bt['DFS'] = adjusted_prices_calculated.loc[adj_close_bt.index, '50642F'].values

[nan 0.6 0.7]
Code
2022-12-30     97.83
2023-01-02     97.83
2023-01-03     96.99
2023-01-04    101.27
2023-01-05     99.55
               ...  
2025-06-24    200.05
2025-06-25    200.05
2025-06-26    200.05
2025-06-27    200.05
2025-06-30    200.05
Name: 50642F, Length: 652, dtype: float64


In [12]:
nan_columns = adj_close_bt.columns[adj_close_bt.isna().any()]
print(nan_columns)

Index([], dtype='object')


In [13]:
adj_close_bt.isna().any().any()

False

In [14]:
adj_close_bt

,AMZN,ABT,AES,IBM,AMD,ADBE,ARE,APD,ALK,BXP,...,HPE,PYPL,VICI,HPQ,DXC,WEC,MNST,LIN,SBAC,CHTR
Date,,,,,,,,,,,,,,,,,,,,,
2022-12-30,84.000000,104.028847,25.521130,126.974380,64.769997,336.529999,129.015121,289.299713,42.939999,57.971348,...,14.839424,71.220001,28.319925,24.539892,26.50,85.590317,50.764999,315.788971,268.495087,339.100006
2023-01-03,85.820000,103.829857,24.944332,127.569229,64.019997,336.920013,127.633484,287.713684,42.160000,57.147842,...,14.932401,74.580002,27.751774,24.430296,27.67,85.955444,50.660000,308.208496,269.864777,341.579987
2023-01-04,85.139999,105.374329,24.207804,128.515503,64.660004,341.410004,130.095657,287.732452,44.299999,57.225048,...,15.267126,77.690002,27.926590,24.859539,28.18,86.521431,51.075001,306.678802,280.295776,354.000000
2023-01-05,83.120003,104.985847,22.956593,127.172661,62.330002,328.440002,126.694679,283.255859,44.110001,55.054764,...,15.295020,76.269997,27.507034,24.850407,27.66,84.357948,50.189999,295.758118,269.510345,361.429993
2023-01-06,86.080002,106.435562,23.453527,129.506866,63.959999,332.750000,130.724487,291.101654,45.279999,57.516697,...,15.908679,76.480003,27.795483,25.900681,28.17,86.512283,51.215000,306.088226,279.165497,367.730011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-24,212.770004,137.462967,10.329869,291.816986,138.429993,382.339996,74.014023,279.216736,49.560001,69.335136,...,18.150000,73.580002,32.810001,24.540001,15.17,105.360001,63.570000,463.160004,236.600006,403.500000
2025-06-25,211.990005,136.785995,10.359468,289.105316,143.399994,387.549988,72.354424,280.210388,48.660000,65.687996,...,18.490000,73.070000,32.259998,24.530001,14.98,102.870003,62.200001,460.200012,232.679993,398.850006
2025-06-26,217.119995,133.072662,10.665318,289.969482,143.679993,384.950012,72.874886,281.204041,49.009998,66.792000,...,18.370001,73.169998,32.189999,24.709999,15.20,103.320000,62.189999,464.459991,230.699997,394.010010


## Calculate historical returns in the period 2023-2025:

In [15]:
def calculate_monthly_returns(df):
    # Clean data: replace zeros and drop fully NaN rows
    df = df.replace(0.0, np.nan).dropna(how="all")
    
    # Resample to monthly frequency using last available price in each month
    df_monthly = df.resample("ME").last()
    
    # Calculate monthly simple returns (percentage change)
    df_returns = df_monthly.pct_change().dropna()
    
    # Optional: set index to mid-month for visualization consistency
    df_returns.index = df_returns.index.map(lambda x: x.replace(day=15))
    
    return df_returns

In [16]:
monthly_returns_23_25 = calculate_monthly_returns(adj_close_bt)

In [17]:
daily_returns_23_25 = adj_close_bt.pct_change().dropna()

## Realized TEs (with 2% TE Constraint) in the period 2023-2025:

In [18]:

with open('Data/optimal_portfolios_1222_without_ceg_ogn_minimum_weight.pkl', 'rb') as f:
    optimal_portfolios_shrink = pickle.load(f)


In [19]:
def compute_buy_and_hold_returns(returns_df, weights, stock_labels):
    """
    Compute buy-and-hold (non-rebalanced) portfolio returns.
    
    Args:
        returns_df: DataFrame with stock returns (monthly), index = date, columns = tickers
        weights: np.array of initial weights (must match stock_labels)
        stock_labels: list or Index of tickers in this sector

    Returns:
        Series of portfolio returns over time (monthly)
    """
    # Subset return data for the sector
    sector_returns = returns_df[stock_labels]
    
    # Compute cumulative returns per stock
    cumulative_returns = (1 + sector_returns).cumprod()
    
    # Multiply by initial weights
    weighted_growth = cumulative_returns.multiply(weights, axis=1)
    
    # Total portfolio value each month
    portfolio_value = weighted_growth.sum(axis=1)
    
    # Convert to monthly portfolio returns
    portfolio_returns = portfolio_value.pct_change().dropna()

    return portfolio_returns

def compute_fixed_weight_returns(returns_df, weights, stock_labels):
    """
    Compute fixed-weight portfolio returns (decarbonised portfolio).
    
    Args:
        returns_df: DataFrame of monthly returns
        weights: np.array of optimised weights
        stock_labels: list or Index of stock tickers

    Returns:
        Series of portfolio returns over time
    """
    sector_returns = returns_df[stock_labels]
    portfolio_returns = sector_returns.dot(weights)
    return portfolio_returns

In [20]:
buy_and_hold_sector_returns = {}
for sector, data in optimal_portfolios_shrink.items():
    w_b = data['w_b_vec']                 # initial benchmark weights
    tickers = data['stock_labels']       # stock labels for the sector
    
    # Compute sector-level buy-and-hold benchmark returns
    sector_returns = compute_buy_and_hold_returns(
        monthly_returns_23_25,
        weights=w_b,
        stock_labels=tickers
    )
    
    buy_and_hold_sector_returns[sector] = sector_returns

buy_and_hold_df = pd.DataFrame(buy_and_hold_sector_returns)



decarb_sector_returns = {}

for sector, data in optimal_portfolios_shrink.items():
    w_opt = data['w_opt']
    tickers = data['stock_labels']
    
    sector_returns = compute_fixed_weight_returns(
        monthly_returns_23_25,
        weights=w_opt,
        stock_labels=tickers
    )
    
    decarb_sector_returns[sector] = sector_returns


sector_tracking_errors = {}

for sector in buy_and_hold_sector_returns:
    # Get returns
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]
    
    # Align indices (just in case)
    common_index = r_b.index.intersection(r_d.index)
    active_returns = r_d.loc[common_index] - r_b.loc[common_index]
    
    # Monthly tracking error
    te_monthly = active_returns.std()
    
    # Annualised tracking error
    te_annualised = te_monthly * np.sqrt(12)
    
    # Store
    sector_tracking_errors[sector] = te_annualised


te_df_total = pd.DataFrame.from_dict(sector_tracking_errors, orient='index', columns=['Annualised Tracking Error'])
te_df_total = te_df_total.sort_values(by='Annualised Tracking Error', ascending=False)
print(te_df_total)

                        Annualised Tracking Error
Information Technology                   0.062288
Communication Services                   0.050658
Health Care                              0.041415
Industrials                              0.039865
Consumer Staples                         0.037306
Consumer Discretionary                   0.036182
Financials                               0.031458
Materials                                0.031355
Real Estate                              0.029201
Utilities                                0.027319
Energy                                   0.019816


Higher tracking error -> for Information Technology, Communication Services and Health Care/Industrials

In [21]:
def expected_shortfall(x: pd.Series, alpha: float = 0.05) -> float:
    """
    ES/CVaR of a return series at level alpha.
    Returns the (negative) mean of the worst alpha tail (i.e., average loss).
    If the series is too short, returns np.nan.
    """
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    var_alpha = np.percentile(x, 100 * alpha)
    tail = x[x <= var_alpha]
    if len(tail) == 0:
        return np.nan
    return tail.mean()  # likely negative for downside

sector_metrics = {}

for sector in buy_and_hold_sector_returns:
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]

    # align
    idx = r_b.index.intersection(r_d.index)
    active = (r_d.loc[idx] - r_b.loc[idx]).dropna()

    if len(active) < 6:
        continue

    # Standard (variance-based) TE
    te_monthly = active.std()
    te_annual = te_monthly * np.sqrt(12)

    # Tail TE (ES / CVaR). Report absolute magnitude for readability.
    es_5 = expected_shortfall(active, alpha=0.05)         # negative number (downside)
    es_1 = expected_shortfall(active, alpha=0.01)         # deeper tail
    es_5_abs = abs(es_5)
    es_1_abs = abs(es_1)

    sector_metrics[sector] = {
        "TE_Annual": te_annual,          # variance-based, annualised
        "ES_5pct_monthly": es_5,         # negative value (avg worst 5% months)
        "ES_1pct_monthly": es_1,         # negative value (avg worst 1% months)
        "ES_5pct_abs": es_5_abs,         # positive magnitude
        "ES_1pct_abs": es_1_abs,
    }

te_df = pd.DataFrame.from_dict(sector_metrics, orient='index')
# Optional formatting in bps / %
print(te_df.sort_values("ES_5pct_monthly", ascending=True))

                        TE_Annual  ES_5pct_monthly  ES_1pct_monthly  \
Information Technology   0.062288        -0.042109        -0.046247   
Communication Services   0.050658        -0.036509        -0.039145   
Industrials              0.039865        -0.025241        -0.027643   
Health Care              0.041415        -0.024569        -0.026281   
Consumer Staples         0.037306        -0.022084        -0.026788   
Materials                0.031355        -0.017445        -0.019360   
Utilities                0.027319        -0.017366        -0.023284   
Consumer Discretionary   0.036182        -0.016414        -0.018885   
Real Estate              0.029201        -0.013658        -0.014514   
Financials               0.031458        -0.013209        -0.013421   
Energy                   0.019816        -0.012538        -0.012800   

                        ES_5pct_abs  ES_1pct_abs  
Information Technology     0.042109     0.046247  
Communication Services     0.036509     0.039

Highest expected shortfall also highest for Info. Tech., Comm. Serv, and Industrials /Health Care

In [22]:

sector_metrics = {}  # New dictionary to store all metrics

rf_monthly = 0.0  # Assumed risk-free rate (adjust if needed)

for sector in buy_and_hold_sector_returns:
    # Get returns
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]
    
    # Align indices
    common_index = r_b.index.intersection(r_d.index)
    r_b = r_b.loc[common_index]
    r_d = r_d.loc[common_index]
    active_returns = r_d - r_b

    # === Tracking Error ===
    te_monthly = active_returns.std()
    te_annualised = te_monthly * np.sqrt(12)

    # === Benchmark metrics ===
    ret_b = r_b.mean() * 12
    vol_b = r_b.std() * np.sqrt(12)
    sharpe_b = (ret_b - rf_monthly * 12) / vol_b if vol_b != 0 else np.nan

    # === Decarbonised metrics ===
    ret_d = r_d.mean() * 12
    vol_d = r_d.std() * np.sqrt(12)
    sharpe_d = (ret_d - rf_monthly * 12) / vol_d if vol_d != 0 else np.nan

    # Store all metrics
    sector_metrics[sector] = {
        "Annualised Tracking Error": round(te_annualised, 3),
        "Return_B": round(ret_b, 2),
        "Vol_B": round(vol_b, 2),
        "Sharpe_B": round(sharpe_b, 2),
        "Return_D": round(ret_d, 2),
        "Vol_D": round(vol_d, 2),
        "Sharpe_D": round(sharpe_d, 2)
    }

# Convert to DataFrame
metrics_df = pd.DataFrame.from_dict(sector_metrics, orient='index')
metrics_df = metrics_df.sort_values(by="Annualised Tracking Error", ascending=False)
print(metrics_df)


                        Annualised Tracking Error  Return_B  Vol_B  Sharpe_B  \
Information Technology                      0.062      0.33   0.19      1.69   
Communication Services                      0.051      0.32   0.17      1.94   
Health Care                                 0.041      0.03   0.13      0.25   
Industrials                                 0.040      0.16   0.16      1.01   
Consumer Staples                            0.037      0.10   0.11      0.85   
Consumer Discretionary                      0.036      0.21   0.20      1.03   
Materials                                   0.031      0.05   0.17      0.31   
Financials                                  0.031      0.19   0.17      1.15   
Real Estate                                 0.029      0.06   0.18      0.35   
Utilities                                   0.027      0.08   0.15      0.52   
Energy                                      0.020      0.03   0.19      0.15   

                        Return_D  Vol_D

Returns of the benchmark almost equal to the returns of the decarb portfolios, except for Inf. Technology and Communication Services even though volatility is slightly higher for the benchmark for these two sectors. The Sharpe Ratio are not too divergent but they differ more for Inf. Technology and Energy. Consumer Discretionary is the only sector with higher Sharpe Ratio for the decarbonised portfolio.

## Testing TEs across 6 months time windows (period 2023-2025), with 2% TE constraint:

In [23]:

sector_tracking_errors = {}
sector_tracking_errors_6m = {}

for sector in buy_and_hold_sector_returns:
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]

    # Align indices
    common_index = r_b.index.intersection(r_d.index)
    r_b = r_b.loc[common_index]
    r_d = r_d.loc[common_index]

    df = pd.DataFrame({'r_b': r_b, 'r_d': r_d})
    df['active_return'] = df['r_d'] - df['r_b']

    six_month_windows = df.resample('6ME')
    te_annualised_list = []

    for _, window in six_month_windows:
        if len(window) >= 3:
            te_6m = window['active_return'].std()
            te_annualised = te_6m * np.sqrt(2)
            te_annualised_list.append(te_annualised)

    sector_tracking_errors_6m[sector] = te_annualised_list
    sector_tracking_errors[sector] = (
        np.mean(te_annualised_list) if te_annualised_list else np.nan
    )

# --- Build Table ---
# Get the max number of 6M periods across all sectors (to align column count)
max_periods = max(len(v) for v in sector_tracking_errors_6m.values())

# Build rows for DataFrame
rows = {}
for sector, te_list in sector_tracking_errors_6m.items():
    row = te_list + [np.nan] * (max_periods - len(te_list))  # pad with NaNs
    row.append(np.mean(te_list) if te_list else np.nan)
    rows[sector] = row

# Create column labels
columns = [f"TE_time_window_{i+1}" for i in range(max_periods)] + ["Avg_TE"]

# Create and display DataFrame
te_df = pd.DataFrame.from_dict(rows, orient='index', columns=columns)
print(te_df.round(4).sort_values(by='Avg_TE', ascending=False))


                        TE_time_window_1  TE_time_window_2  TE_time_window_3  \
Information Technology            0.0132            0.0235            0.0384   
Communication Services            0.0073            0.0194            0.0240   
Health Care                       0.0084            0.0194            0.0244   
Industrials                       0.0149            0.0075            0.0102   
Consumer Staples                  0.0059            0.0124            0.0187   
Consumer Discretionary            0.0117            0.0096            0.0171   
Financials                        0.0113            0.0152            0.0129   
Real Estate                       0.0094            0.0092            0.0084   
Materials                         0.0074            0.0159            0.0098   
Utilities                         0.0051            0.0103            0.0060   
Energy                            0.0060            0.0093            0.0070   

                        TE_time_window_

The TE averaged across 6 months time windows is significantly lower (in the range less than 1% to 2.5%) than the TE calculated over the entire period (between 2%-6%). The difference is especially high for the three sectors with the highest TE. 

In [24]:

# Containers
sector_tracking_errors_6m = {}
sharpe_ratios_b_6m = {}
sharpe_ratios_d_6m = {}
returns_b_6m = {}
returns_d_6m = {}
vol_b_6m = {}
vol_d_6m = {}

summary_metrics = {}

for sector in buy_and_hold_sector_returns:
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]

    # Align indices
    common_index = r_b.index.intersection(r_d.index)
    r_b = r_b.loc[common_index]
    r_d = r_d.loc[common_index]

    df = pd.DataFrame({'r_b': r_b, 'r_d': r_d})
    df['active_return'] = df['r_d'] - df['r_b']

    six_month_windows = df.resample('6ME')

    te_annualised_list = []
    sharpe_b_list = []
    sharpe_d_list = []
    return_b_list = []
    return_d_list = []
    vol_b_list = []
    vol_d_list = []

    for _, window in six_month_windows:
        if len(window) >= 3:
            # --- Tracking Error ---
            te_6m = window['active_return'].std()
            te_annualised = te_6m * np.sqrt(2)
            te_annualised_list.append(te_annualised)

            # --- Benchmark Metrics ---
            r_b_win = window['r_b']
            mean_b = r_b_win.mean() * 12
            std_b = r_b_win.std() * np.sqrt(12)
            sharpe_b = mean_b / std_b if std_b != 0 else np.nan
            return_b_list.append(mean_b)
            vol_b_list.append(std_b)
            sharpe_b_list.append(sharpe_b)

            # --- Decarb Metrics ---
            r_d_win = window['r_d']
            mean_d = r_d_win.mean() * 12
            std_d = r_d_win.std() * np.sqrt(12)
            sharpe_d = mean_d / std_d if std_d != 0 else np.nan
            return_d_list.append(mean_d)
            vol_d_list.append(std_d)
            sharpe_d_list.append(sharpe_d)

    # Store lists per sector
    sector_tracking_errors_6m[sector] = te_annualised_list
    returns_b_6m[sector] = return_b_list
    returns_d_6m[sector] = return_d_list
    vol_b_6m[sector] = vol_b_list
    vol_d_6m[sector] = vol_d_list
    sharpe_ratios_b_6m[sector] = sharpe_b_list
    sharpe_ratios_d_6m[sector] = sharpe_d_list

    # Store averaged metrics in summary table
    summary_metrics[sector] = {
        "Avg_TE": np.nanmean(te_annualised_list),
        "Avg_Return_B": np.nanmean(return_b_list),
        "Avg_Return_D": np.nanmean(return_d_list),
        "Avg_Vol_B": np.nanmean(vol_b_list),
        "Avg_Vol_D": np.nanmean(vol_d_list),
        "Avg_Sharpe_B": np.nanmean(sharpe_b_list),
        "Avg_Sharpe_D": np.nanmean(sharpe_d_list)
    }

# Get max number of 6M periods
max_periods = max(len(v) for v in sector_tracking_errors_6m.values())

# Build detailed rows
detailed_rows = {}

for sector in sector_tracking_errors_6m:
    row = {}

    # TE per 6M
    te_list = sector_tracking_errors_6m[sector]
    for i, val in enumerate(te_list):
        row[f"TE_{i+1}"] = val
    row["Avg_TE"] = np.nanmean(te_list)

    # Returns Benchmark
    ret_b_list = returns_b_6m[sector]
    for i, val in enumerate(ret_b_list):
        row[f"Return_B_{i+1}"] = val
    row["Avg_Return_B"] = np.nanmean(ret_b_list)

    # Returns Decarb
    ret_d_list = returns_d_6m[sector]
    for i, val in enumerate(ret_d_list):
        row[f"Return_D_{i+1}"] = val
    row["Avg_Return_D"] = np.nanmean(ret_d_list)

    # Volatility Benchmark
    vol_b_list = vol_b_6m[sector]
    for i, val in enumerate(vol_b_list):
        row[f"Vol_B_{i+1}"] = val
    row["Avg_Vol_B"] = np.nanmean(vol_b_list)

    # Volatility Decarb
    vol_d_list = vol_d_6m[sector]
    for i, val in enumerate(vol_d_list):
        row[f"Vol_D_{i+1}"] = val
    row["Avg_Vol_D"] = np.nanmean(vol_d_list)

    # Sharpe Benchmark
    sharpe_b_list = sharpe_ratios_b_6m[sector]
    for i, val in enumerate(sharpe_b_list):
        row[f"Sharpe_B_{i+1}"] = val
    row["Avg_Sharpe_B"] = np.nanmean(sharpe_b_list)

    # Sharpe Decarb
    sharpe_d_list = sharpe_ratios_d_6m[sector]
    for i, val in enumerate(sharpe_d_list):
        row[f"Sharpe_D_{i+1}"] = val
    row["Avg_Sharpe_D"] = np.nanmean(sharpe_d_list)

    detailed_rows[sector] = row
# Create full table
detailed_df = pd.DataFrame.from_dict(detailed_rows, orient='index')

# Round columns selectively
rounded_df = detailed_df.copy()
for col in rounded_df.columns:
    if col.startswith("TE_") or col == "Avg_TE":
        rounded_df[col] = rounded_df[col].round(3)
    else:
        rounded_df[col] = rounded_df[col].round(2)

# Optional: sort by average Sharpe_D
rounded_df = rounded_df.sort_values(by="Avg_Sharpe_D", ascending=False)

# Display the full table
pd.set_option('display.max_columns', None)
print(rounded_df)


                         TE_1   TE_2   TE_3   TE_4   TE_5  Avg_TE  Return_B_1  \
Communication Services  0.007  0.019  0.024  0.022  0.031   0.021        0.60   
Information Technology  0.013  0.024  0.038  0.021  0.032   0.026        0.59   
Consumer Discretionary  0.012  0.010  0.017  0.024  0.013   0.015        0.38   
Financials              0.011  0.015  0.013  0.009  0.018   0.013       -0.01   
Industrials             0.015  0.007  0.010  0.022  0.024   0.016        0.16   
Utilities               0.005  0.010  0.006  0.012  0.016   0.010       -0.04   
Consumer Staples        0.006  0.012  0.019  0.014  0.026   0.015        0.07   
Materials               0.007  0.016  0.010  0.013  0.010   0.011        0.07   
Energy                  0.006  0.009  0.007  0.010  0.012   0.009        0.17   
Health Care             0.008  0.019  0.024  0.019  0.015   0.017        0.12   
Real Estate             0.009  0.009  0.008  0.016  0.019   0.012       -0.01   

                        Ret

In [25]:
summary_df = pd.DataFrame.from_dict(summary_metrics, orient='index')

# Optional: round and sort
summary_df = summary_df.round({
    "Avg_TE": 3,
    "Avg_Return_B": 2,
    "Avg_Return_D": 2,
    "Avg_Vol_B": 2,
    "Avg_Vol_D": 2,
    "Avg_Sharpe_B": 2,
    "Avg_Sharpe_D": 2
})

summary_df = summary_df.sort_values(by="Avg_Sharpe_D", ascending=False)

# Display full columns
pd.set_option('display.max_columns', None)
print(summary_df)


                        Avg_TE  Avg_Return_B  Avg_Return_D  Avg_Vol_B  \
Communication Services   0.021          0.35          0.29       0.17   
Information Technology   0.026          0.34          0.24       0.20   
Consumer Discretionary   0.015          0.21          0.21       0.21   
Financials               0.013          0.20          0.20       0.17   
Industrials              0.016          0.18          0.14       0.18   
Utilities                0.010          0.10          0.10       0.14   
Consumer Staples         0.015          0.10          0.06       0.11   
Materials                0.011          0.06          0.06       0.17   
Energy                   0.009          0.05          0.02       0.20   
Health Care              0.017          0.03         -0.00       0.13   
Real Estate              0.012          0.08          0.07       0.16   

                        Avg_Vol_D  Avg_Sharpe_B  Avg_Sharpe_D  
Communication Services       0.14          2.22          2.

The Sharpe Ratio is always higher for the benchmark, especially for Consumer Staple and Health Care. The only exception where the Sharpe Ratio is higher for the decarbonised portfolio is Consumer Discretionary and Materials. 

## Realized TE across different TE objectives

In [26]:
with open("Data/optimal_portfolios_by_te.pkl", "rb") as f:
    optimal_portfolios_by_te = pickle.load(f)

In [27]:
exclude_list = ['DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA']

# List of result rows
results = []

for te_cap, sector_data in optimal_portfolios_by_te.items():
    for sector, data in sector_data.items():
        stock_labels = data['stock_labels']
        w_b = data['w_b_vec']
        w_opt = data['w_opt']
        
 
        # Compute returns
        r_b = compute_buy_and_hold_returns(
            monthly_returns_23_25,
            weights=w_b,
            stock_labels=stock_labels
        )

        r_d = compute_fixed_weight_returns(
            monthly_returns_23_25,
            weights=w_opt,
            stock_labels=stock_labels
        )

        # Align indices
        common_index = r_b.index.intersection(r_d.index)
        r_b = r_b.loc[common_index]
        r_d = r_d.loc[common_index]
        active_returns = r_d - r_b

        # TE calculation
        te_monthly = active_returns.std()
        te_annualised = te_monthly * np.sqrt(12)

        # Benchmark stats
        ret_b = r_b.mean() * 12
        vol_b = r_b.std() * np.sqrt(12)
        sharpe_b = ret_b / vol_b if vol_b != 0 else np.nan

        # Optimised stats
        ret_d = r_d.mean() * 12
        vol_d = r_d.std() * np.sqrt(12)
        sharpe_d = ret_d / vol_d if vol_d != 0 else np.nan

        # Store as row
        results.append({
            "Sector": sector,
            "TE_Cap": te_cap,
            "TE_Annualised": te_annualised,
            "Return_B": ret_b,
            "Return_D": ret_d,
            "Vol_B": vol_b,
            "Vol_D": vol_d,
            "Sharpe_B": sharpe_b,
            "Sharpe_D": sharpe_d
        })


df = pd.DataFrame(results)
# Round columns as requested
df["TE_Annualised"] = df["TE_Annualised"].round(3)
df[["Return_B", "Return_D", "Sharpe_B", "Sharpe_D"]] = df[["Return_B", "Return_D", "Sharpe_B", "Sharpe_D"]].round(2)

for te in df['TE_Cap'].unique():
    df_filtered = df.loc[df['TE_Cap'] == te]
    # Sort and display
    df_filtered = df_filtered.sort_values(by=["TE_Annualised"], ascending=[False])
    print(df_filtered.loc[:, ['Sector', 'TE_Cap', 'TE_Annualised', 'Sharpe_B', 'Sharpe_D']])
    print()


                    Sector  TE_Cap  TE_Annualised  Sharpe_B  Sharpe_D
9   Communication Services    0.01          0.046      1.94      1.98
3   Information Technology    0.01          0.045      1.69      1.53
0   Consumer Discretionary    0.01          0.037      1.03      1.06
1              Health Care    0.01          0.032      0.25      0.03
6              Industrials    0.01          0.030      1.01      0.83
10        Consumer Staples    0.01          0.029      0.85      0.56
4              Real Estate    0.01          0.023      0.35      0.28
5                Materials    0.01          0.023      0.31      0.25
7               Financials    0.01          0.017      1.15      1.14
2                Utilities    0.01          0.016      0.52      0.48
8                   Energy    0.01          0.014      0.15      0.03

                    Sector  TE_Cap  TE_Annualised  Sharpe_B  Sharpe_D
14  Information Technology    0.02          0.062      1.69      1.43
20  Communication S